In [0]:
%sh
nc -zv mysql-catalog-ue-np-1.mysql.database.azure.com 3306

Connection to mysql-catalog-ue-np-1.mysql.database.azure.com (10.176.152.8) 3306 port [tcp/mysql] succeeded!


In [0]:
%sh nslookup mysql-catalog-ue-np-1.mysql.database.azure.com

Server:		168.63.129.16
Address:	168.63.129.16#53

Non-authoritative answer:
mysql-catalog-ue-np-1.mysql.database.azure.com	canonical name = mysql-catalog-ue-np-1.privatelink.mysql.database.azure.com.
Name:	mysql-catalog-ue-np-1.privatelink.mysql.database.azure.com
Address: 10.176.152.8



In [0]:
from pyspark.sql.functions import lit
from databricks.sdk.runtime import dbutils

from datetime import datetime

# --- Parameters (widgets with dev defaults; overridden by job parameters in prod) ---
caddyshack_user_prod = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="caddyshack_user_prod")
caddyshack_password_prod = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="caddyshack_password_prod")

dbutils.widgets.text("catalog_name", "dev_sdsc_db", "Catalog")
dbutils.widgets.text("bronze_schema_name", "sdds_bronze", "Bronze Schema")

# MySQL connection parameters
# Full JDBC URL, e.g. jdbc:mysql://<host>:<port>/<database>
dbutils.widgets.text("mysql_jdbc_url", "jdbc:mysql://mysql-catalog-ue-np-1.mysql.database.azure.com", "MySQL JDBC URL")
dbutils.widgets.text("extraction_date", datetime.now().strftime("%Y-%m-%d"), "Extraction Date")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("bronze_schema_name")
unpub_cat_table_name = 'unpublished_category_events'
del_cat_table_name = 'deleted_category_events'

#secret_scope = dbutils.widgets.get("secret_scope")
mysql_jdbc_url = dbutils.widgets.get("mysql_jdbc_url")
extraction_date = dbutils.widgets.get("extraction_date")

# --- Credentials pulled from the secret scope (never hardcode secrets) ---
# Store your MySQL credentials in the Databricks secret scope defined above.
#   databricks secrets put-secret <secret_scope> mysql-user
#   databricks secrets put-secret <secret_scope> mysql-password
#mysql_user = dbutils.secrets.get(scope=secret_scope, key="mysql-user")
#mysql_password = dbutils.secrets.get(scope=secret_scope, key="mysql-password")
#mysql_user = dbutils.widgets.get("mysql_user")
#mysql_password = dbutils.widgets.get("mysql_password")

mysql_user = caddyshack_user_prod
mysql_password = caddyshack_password_prod


In [0]:
def unpublished_cat_query():
    """Return the SQL query used to extract category records from MySQL."""
    return "select * from service_instance_db.category_desc where category_id != -1 and published = false"

def deleted_cat_query():
    """Return the SQL query used to extract category records from MySQL."""
    return "select * from service_instance_db.category where category_id != -1 and deleted = true"

def read_mysql_query(jdbc_url, query, user, password):
    """Read the result of a SQL query from MySQL into a Spark DataFrame via JDBC (raw bronze ingestion)."""
    return (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", query)
        .option("user", user)
        .option("password", password)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .load()
    )


def add_load_metadata(df, load_timestamp, extraction_date):
    """Attach load metadata columns to the bronze DataFrame."""
    return (
        df
        .withColumn("load_timestamp", lit(load_timestamp).cast("timestamp"))
        .withColumn("extraction_date", lit(extraction_date).cast("date"))
    )


# --- Orchestration: extract from MySQL, then attach bronze load metadata ---
load_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

unpub_df = read_mysql_query(mysql_jdbc_url, unpublished_cat_query(), mysql_user, mysql_password)
del_df = read_mysql_query(mysql_jdbc_url, deleted_cat_query(), mysql_user, mysql_password)

unpub_df = add_load_metadata(unpub_df, load_timestamp, extraction_date)
del_df= add_load_metadata(del_df, load_timestamp, extraction_date)

record_count_no_pub = unpub_df.count()
record_count_del_df = del_df.count()
print(f"Extracted {record_count_no_pub} records from MySQL no pub cat")
print(f"Extracted {record_count_del_df} records from MySQL del del cat")

Extracted 158901 records from MySQL no pub cat
Extracted 84697 records from MySQL del del cat


In [0]:
unpub_df.limit(10).display()

category_id,language_id,description,fullimage,keyword,name,published,sort_order,thumbnail,load_timestamp,extraction_date
63051,-1,DUMMY ASSET,null,null,DUMMYASSET,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63207,-1,600-XXX-004-001,null,null,Fitness Tops,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63208,-1,600-XXX-004-002,null,null,Fitness Bottoms,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63209,-1,600-XXX-007-001,null,null,Soccer Tops,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63214,-1,690-001-XXX-002,null,null,Men's Sack Packs,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63215,-1,700-XXX-003-002,null,null,Fleece Bottoms,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63216,-1,700-XXX-013-002,null,null,Other Bottoms,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63422,-1,165-XXX-001,null,null,Women's Golf Tops,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63423,-1,165-XXX-002,null,null,Women's Golf Bottoms,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10
63538,-1,600-XXX-008-002,null,null,Lacrosse Bottoms,false,MRA,null,2026-08-10T23:41:31Z,2026-08-10


In [0]:
del_df.display()

category_id,deleted,update_count,update_date_time,update_user,catalog_id,flyout,hyper_category_type,identifier,search_type,type,needs_review,normalized_name,create_date_time,create_user,load_timestamp,extraction_date
76718,true,894,2023-03-23T13:37:04.555762Z,Nathan.Milliren,10051,true,NH,Sweaters-112285,SA,D,false,golf sweaters women,2023-03-23T13:37:04.555762Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76753,true,894,2023-03-23T13:37:04.555778Z,Nathan.Milliren,10051,true,NH,Sweaters-112320,SA,D,false,golf men sweaters,2023-03-23T13:37:04.555778Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76781,true,895,2023-03-23T13:37:04.555784Z,Nathan.Milliren,10051,true,NH,GolfBalls-112348,SA,D,false,balls golf,2023-03-23T13:37:04.555784Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76787,true,892,2023-03-23T13:37:04.555789Z,Nathan.Milliren,10051,true,NH,TeamGolfBalls-112354,SA,D,false,balls golf team,2023-03-23T13:37:04.555789Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76797,true,894,2023-03-23T13:37:04.555795Z,Nathan.Milliren,10051,false,NH,BagAccessories-112364,SA,D,false,accessories bag,2023-03-23T13:37:04.555795Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76798,true,894,2023-03-23T13:37:04.5558Z,Nathan.Milliren,10051,true,NH,Carts-112365,SA,D,false,carts,2023-03-23T13:37:04.5558Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76799,true,894,2023-03-23T13:37:04.555806Z,Nathan.Milliren,10051,true,NH,PushPullCarts-112366,SA,D,false,carts pull push,2023-03-23T13:37:04.555806Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76800,true,894,2023-03-23T13:37:04.555934Z,Nathan.Milliren,10051,true,NH,CartAccessories-112367,SA,D,false,accessories cart,2023-03-23T13:37:04.555934Z,Nathan.Milliren,2026-08-10T23:41:31Z,2026-08-10
76823,true,2130,2025-02-03T18:54:39.446011Z,Caitlin.Phalon,10051,false,NH,TeamAccessories-112390,NN,D,false,accessories team,2022-06-27T01:08:31.351593Z,srv-batch-ecommsearch [null],2026-08-10T23:41:31Z,2026-08-10
76831,true,2121,2025-01-28T20:46:39.36198Z,caitlin.phalon,10051,true,NH,Socks-112398,NN,D,false,golf socks,2022-12-01T11:10:58.61651Z,srv-batch-ecommsearch [null],2026-08-10T23:41:31Z,2026-08-10


In [0]:
# --- Step 3: Write extraction data to Delta table, partitioned by extraction_date (idempotent) ---
def write_bronze_delta(df, catalog_name, schema_name, table_name, extraction_date, partition_col="extraction_date"):
    """Idempotently write the bronze DataFrame to a partitioned Delta table.

    First run creates the table; re-runs overwrite only the current ``extraction_date`` partition.
    Legacy tables partitioned by a different (or no) column are dropped and recreated.
    Returns the fully-qualified target table name.
    """
    target_table = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"
    table_exists = spark.catalog.tableExists(f"{catalog_name}.{schema_name}.`{table_name}`")

    # Migration: drop table if it exists with wrong (or no) partition column
    if table_exists:
        partition_cols = spark.sql(f"DESCRIBE DETAIL {target_table}").select("partitionColumns").collect()[0][0]
        if partition_col not in str(partition_cols):
            spark.sql(f"DROP TABLE IF EXISTS {target_table}")
            table_exists = False
            print(f"Dropped {target_table} \u2014 wrong partition ({partition_cols}), will recreate with {partition_col}")

    if not table_exists:
        # First run: create table with extraction_date partition
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy(partition_col)
            .saveAsTable(target_table)
        )
    else:
        # Subsequent runs: overwrite only this partition (idempotent re-runs)
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"{partition_col} = '{extraction_date}'")
            .partitionBy(partition_col)
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
    return target_table

unpub_cat_table = write_bronze_delta(unpub_df, catalog_name, schema_name, unpub_cat_table_name, extraction_date)
del_cat_table = write_bronze_delta(del_df, catalog_name, schema_name, del_cat_table_name, extraction_date)

record_count_unpub_cat_table = unpub_df.count()
record_count_del_cat_table = del_df.count()

print(f"Wrote {record_count_unpub_cat_table} records to {unpub_cat_table_name} (partition: extraction_date = '{extraction_date}')")
print(f"Wrote {record_count_del_cat_table} records to {del_cat_table_name} (partition: extraction_date = '{extraction_date}')")

Wrote 158901 records to unpublished_category_events (partition: extraction_date = '2026-08-10')
Wrote 84697 records to deleted_category_events (partition: extraction_date = '2026-08-10')
